# IntelliDoc AI — Notebook 4: CORD Dataset Validation on Google Colab

**Purpose:** Load the CORD (Consolidated Receipt Dataset) on Google Colab, inspect its 30 fine-grained financial line-item labels, visualize receipt bounding boxes, and evaluate real-world receipt arithmetic consistency.

---
### Before Running This Notebook in Google Colab
1. Upload your unzipped `cord/` folder to your **Google Drive** inside `MyDrive/`.
2. Your Google Drive structure should look like:
```
MyDrive/
    cord/
        train/
            image/
            json/
        dev/
            image/
            json/
        test/
            image/
            json/
```
*(Alternatively, if you have `cord.zip`, you can upload it to Google Drive and unzip it directly in Step 1!)*

## Step 1: Mount Google Drive
Connect your Google Drive so Colab can read your CORD files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')

## Step 2: Set CORD Dataset Path
Verify that the CORD folder and splits exist on your Drive.

In [ ]:
import os
import zipfile

# Check common Google Drive locations
possible_paths = [
    '/content/drive/MyDrive/Intellidoc_AI/datasets_public/cord',
    '/content/drive/MyDrive/cord',
    '/content/drive/MyDrive/CORD',
    '/content/drive/MyDrive/cord-v0',
    '/content/cord'
]

CORD_ROOT = None
for p in possible_paths:
    if os.path.exists(p):
        CORD_ROOT = p
        break

# If not found as folder, check if a zip file was uploaded to MyDrive
if CORD_ROOT is None:
    zip_path = '/content/drive/MyDrive/cord.zip'
    if os.path.exists(zip_path):
        print(f'Extracting {zip_path} to /content/cord...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/cord')
        CORD_ROOT = '/content/cord'

if CORD_ROOT is None:
    CORD_ROOT = '/content/drive/MyDrive/cord'

TRAIN_DIR = os.path.join(CORD_ROOT, 'train')
DEV_DIR   = os.path.join(CORD_ROOT, 'dev')
TEST_DIR  = os.path.join(CORD_ROOT, 'test')

print(f'CORD Root Directory: {CORD_ROOT}')
for name, path in [('Train', TRAIN_DIR), ('Dev', DEV_DIR), ('Test', TEST_DIR)]:
    exists = os.path.exists(path)
    print(f'  {"✅ EXISTS" if exists else "❌ MISSING"} → {name}: {path}')

In [ ]:
# Install required lightweight visualization libraries
!pip install -q Pillow matplotlib pandas
print('✅ Required libraries ready!')

## Step 3: Load CORD Annotations
Load all JSON annotation files from `train/json/`, `dev/json/`, and `test/json/`.

In [ ]:
import json
import glob

def load_cord_split(split_dir):
    """Loads all annotation JSON files from a CORD split."""
    json_dir = os.path.join(split_dir, 'json')
    json_files = sorted(glob.glob(os.path.join(json_dir, '*.json')))
    receipts = []
    for filepath in json_files:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                data['filename'] = os.path.basename(filepath).replace('.json', '')
                receipts.append(data)
        except Exception as e:
            pass
    return receipts

train_receipts = load_cord_split(TRAIN_DIR)
dev_receipts   = load_cord_split(DEV_DIR)
test_receipts  = load_cord_split(TEST_DIR)

print(f'Training receipts loaded : {len(train_receipts)}')
print(f'Dev receipts loaded      : {len(dev_receipts)}')
print(f'Test receipts loaded     : {len(test_receipts)}')
print(f'Total Receipts           : {len(train_receipts) + len(dev_receipts) + len(test_receipts)}')

## Step 4: CORD Semantic Category Distribution
Analyze the distribution of CORD's **30 fine-grained semantic categories** (menu items, prices, subtotals, tax, and total).

In [ ]:
from collections import Counter
import pandas as pd

category_counter = Counter()
total_tokens = 0

all_receipts = train_receipts + dev_receipts + test_receipts
for r in all_receipts:
    for line in r.get('valid_line', []):
        cat = line.get('category', 'other')
        category_counter[cat] += 1
        total_tokens += len(line.get('words', []))

print(f'Total Valid Lines Annotated : {sum(category_counter.values())}')
print(f'Total Word Tokens           : {total_tokens}')
print(f'Unique Categories           : {len(category_counter)}')

df_cats = pd.DataFrame(category_counter.most_common(20), columns=['Category', 'Count'])
df_cats['Percentage'] = (df_cats['Count'] / sum(category_counter.values()) * 100).round(2).astype(str) + '%'
display(df_cats)

## Step 5: Inspect One Raw Receipt Annotation
View the JSON structure of a sample CORD receipt.

In [ ]:
import pprint

if test_receipts:
    sample = test_receipts[0]
    print(f"Receipt ID: {sample['filename']}")
    print(f"Valid Line Count: {len(sample.get('valid_line', []))}")
    print("\nFirst 3 annotated lines:")
    pprint.pprint(sample.get('valid_line', [])[:3])

## Step 6: Visualize Receipt with Bounding Boxes
Draw colored bounding box overlays on the receipt image based on semantic category.

**Color Legend:**
- 🔵 **Blue** = `menu.*` (Menu item name, quantity, unit price, line price)
- 🟢 **Green** = `sub_total.*` (Subtotal, Tax, Service charge)
- 🔴 **Red** = `total.*` (Total price, Cash tender, Change)
- 🟠 **Orange** = `void_total.*` (Cancelled/Void items)

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def get_cat_color(cat):
    if 'menu' in cat:
        return (30, 144, 255, 120)    # Blue
    elif 'sub_total' in cat:
        return (46, 204, 113, 120)   # Green
    elif 'total' in cat:
        return (231, 76, 60, 120)    # Red
    else:
        return (243, 156, 18, 120)   # Orange

def visualize_cord_receipt(receipt, split_dir):
    filename = receipt['filename']
    # Check for image with .png or .jpg
    img_path = os.path.join(split_dir, 'image', filename + '.png')
    if not os.path.exists(img_path):
        img_path = os.path.join(split_dir, 'image', filename + '.jpg')
        
    if not os.path.exists(img_path):
        print(f'Image not found: {img_path}')
        return

    img = Image.open(img_path).convert('RGBA')
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    for line in receipt.get('valid_line', []):
        cat = line.get('category', 'other')
        color = get_cat_color(cat)
        for word in line.get('words', []):
            q = word.get('quad', {})
            if q:
                pts = [(q['x1'], q['y1']), (q['x2'], q['y2']), (q['x3'], q['y3']), (q['x4'], q['y4'])]
                draw.polygon(pts, fill=color, outline=color[:3] + (255,))

    combined = Image.alpha_composite(img, overlay).convert('RGB')

    plt.figure(figsize=(10, 14))
    plt.imshow(combined)
    plt.title(f'CORD Receipt: {filename} | Blue=Menu, Green=Subtotal/Tax, Red=Total', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Display first test receipt
if test_receipts:
    visualize_cord_receipt(test_receipts[0], TEST_DIR)

## Step 7: IntelliDoc Deterministic Arithmetic Audit on CORD Receipts
Apply IntelliDoc's verification rule on real CORD receipts: check if the sum of all `menu.price` line items equals the stated `total.total_price` (accounting for `sub_total.tax_price`).

In [ ]:
import re

def parse_price(val_str):
    clean = re.sub(r'[^0-9.]', '', val_str.replace(',', '.'))
    try:
        return float(clean)
    except Exception:
        return None

audit_cases = []
for r in test_receipts[:30]:
    line_totals = []
    stated_tax = 0.0
    stated_total = None
    
    for line in r.get('valid_line', []):
        cat = line.get('category', '')
        text = " ".join([w.get('text', '') for w in line.get('words', []) if w.get('is_key') == 0])
        
        if cat == 'menu.price':
            p = parse_price(text)
            if p is not None:
                line_totals.append(p)
        elif cat == 'sub_total.tax_price':
            t = parse_price(text)
            if t is not None:
                stated_tax = t
        elif cat == 'total.total_price':
            tot = parse_price(text)
            if tot is not None:
                stated_total = tot

    if line_totals and stated_total:
        calc_sum = sum(line_totals)
        diff = abs((calc_sum + stated_tax) - stated_total)
        verdict = "VERIFIED" if diff <= 1.0 else "REQUIRES_REVIEW"
        audit_cases.append({
            'Receipt': r['filename'],
            'Items': len(line_totals),
            'Calculated Sum': f"{calc_sum:,.2f}",
            'Tax': f"{stated_tax:,.2f}",
            'Stated Total': f"{stated_total:,.2f}",
            'Difference': f"{diff:,.2f}",
            'Verdict': verdict
        })

df_results = pd.DataFrame(audit_cases)
print(f'Audited {len(df_results)} CORD Receipts with IntelliDoc Zero-Hallucination Invariant Engine:')
display(df_results.head(15))

print('\nVerification Verdict Distribution:')
print(df_results['Verdict'].value_counts())

## ✅ Validation Complete

If you see:
- Receipts loaded across `train`, `dev`, and `test` splits
- 30 semantic categories breakdown
- Color-coded bounding box overlays on receipt scans
- IntelliDoc deterministic arithmetic proofs (`VERIFIED` vs `REQUIRES_REVIEW`)

Your CORD dataset is **100% validated and ready** for fine-tuning and evaluation in IntelliDoc AI!